In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("Lines", StringType()),
    StructField("Lon", DoubleType()),
    StructField("VehicleNumber", StringType()),
    StructField("Time", StringType()),  # Możesz później przekonwertować na TimestampType
    StructField("Lat", DoubleType()),
    StructField("Brigade", StringType())
])

In [0]:
from pyspark.sql.functions import *
import time

streaming_df = (spark
  .readStream
  .format("json")
  .schema(schema)
  .option("multiLine", "true")
  .option("pathGlobFilter", "*.json")
  .option("recursiveFileLookup", "true")
  .option("maxFilesPerTrigger", 50)
  .load("dbfs:/mnt/busdataapi/")
)
processed_df = streaming_df.withColumn(
    "Timestamp", to_timestamp(col("Time"), "yyyy-MM-dd HH:mm:ss")
).drop("Time")

query = (processed_df
  .writeStream
  .trigger(once=True)
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", "/mnt/busdataapi/checkpoints")
  .option("mergeSchema", "true")
  .trigger(once=True)
  .start("/mnt/busdataapi/bus_data_delta")
)